# 07 — Risk and Position Sizing

**Strategy reference:** §14.4 (Sizing formula), §15 (Risk
budgeting), §14.1–14.3 (Scale-in / Scale-out / Trailing stop).

Three-step sizing:

1. **Risk-denominated raw size**
   $Q_{\text{risk}} = \text{target\_risk\_usd} / |P_{\text{entry}} - P_{\text{stop}}|$
2. **Volatility adjustment**
   $Q_{\text{raw}} = Q_{\text{risk}} \cdot \sigma_{\text{target}} / \hat\sigma_{\text{realized}}$
   clipped to $[0.25, 2.0]$.
3. **Tide throttle and budget clip**
   $Q_{\text{final}} = \text{clip}\,(Q_{\text{raw}} \cdot \text{risk\_mult} \cdot R_{\text{budget}}, 0, Q_{\max})$

In [ ]:
# ── Data-source configuration ─────────────────────────────────────────
# OHLCV (Parquet) — S3 or local, controlled by DATA_STORE env var:
#   Local (default):  reads <project_root>/data/ohlcv/...
#   S3:               uncomment the two lines below
# import os
# os.environ["DATA_STORE"] = "s3"
# os.environ["S3_BUCKET"]  = "trading-data-centheos"
#
# Tick data (HDF5) — always stored locally; pull from S3 on demand:
#   load_ticks(...)              → use local cache (fast, no network)
#   load_ticks(..., refresh=True) → sync from S3 then read (ETag-gated)
#   Requires: AWS_PROFILE=trading (or AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY)
import os
os.environ["AWS_PROFILE"] = "trading"
os.environ["S3_BUCKET"]   = "trading-data-centheos"
# ─────────────────────────────────────────────────────────────────────

import sys, importlib
from pathlib import Path

_here = Path.cwd().resolve()
for _cand in [_here, *_here.parents]:
    if (_cand / "schemas.py").exists():
        _root = _cand; break
else:
    raise RuntimeError("Could not locate project root (no schemas.py found)")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import notebooks.utils as _utils_mod
importlib.reload(_utils_mod)   # always pick up on-disk changes without restarting the kernel

from notebooks.utils import (
    load_ohlcv, list_ohlcv, load_ticks, latest_book,
    plot_ohlcv, plot_equity_curve, configure_pandas, env_summary,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

configure_pandas()
%matplotlib inline

In [ ]:
from schemas import RiskConfig, RippleConfig
rc = RiskConfig()
rip = RippleConfig()
print('target_risk_usd =', rc.target_risk_usd)
print('sigma_target    =', rc.sigma_target)
print('vol_ratio_clamp =', rip.vol_ratio_clamp)

## 1. Sizing function

In [ ]:
def position_size(entry_price, stop_price, sigma_realized, *,
                  risk_multiplier, consumed_es, es_budget,
                  max_position_usd, cfg_risk=rc, cfg_ripple=rip):
    distance = abs(entry_price - stop_price)
    if distance <= 0:
        return 0.0
    Q_risk = cfg_risk.target_risk_usd / distance
    vol_ratio = cfg_risk.sigma_target / max(sigma_realized, 1e-6)
    lo, hi = cfg_ripple.vol_ratio_clamp
    vol_ratio = max(lo, min(vol_ratio, hi))
    Q_raw = Q_risk * vol_ratio
    R_budget = max(0.0, min((es_budget - consumed_es) / max(es_budget, 1e-9), 1.0))
    Q_max = max_position_usd / entry_price
    return float(min(max(Q_raw * risk_multiplier * R_budget, 0.0), Q_max))

position_size(67_000, 66_500, 0.6, risk_multiplier=1.0,
              consumed_es=0.0, es_budget=1000.0, max_position_usd=10_000.0)

## 2. Sensitivity surface
Hold entry / stop / max_position fixed; sweep σ̂ and the Tide
risk multiplier.

In [ ]:
sigma_grid = np.linspace(0.1, 2.0, 60)
rm_grid    = np.linspace(0.0, 1.0, 30)
Z = np.zeros((rm_grid.size, sigma_grid.size))
for i, rm in enumerate(rm_grid):
    for j, sig in enumerate(sigma_grid):
        Z[i, j] = position_size(67_000, 66_500, sig,
                                 risk_multiplier=rm, consumed_es=0.0,
                                 es_budget=1000.0, max_position_usd=10_000.0)

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(Z, aspect='auto', origin='lower',
                extent=[sigma_grid[0], sigma_grid[-1], rm_grid[0], rm_grid[-1]],
                cmap='viridis')
ax.set_xlabel('σ̂_realized'); ax.set_ylabel('risk_multiplier')
ax.set_title('Q_final (units of base) vs σ̂ and risk_multiplier')
plt.colorbar(im, label='Q_final')
plt.show()

## 3. Budget consumption over a sequence of trades

In [ ]:
ES_MULT_95 = 0.10314211381526195 / 0.05   # φ(z_.95)/(1-.95) ≈ 2.063
def parametric_es(notional, sigma_ann, horizon_years=1/365, mult=ES_MULT_95):
    return notional * sigma_ann * np.sqrt(horizon_years) * mult

es_budget = 100.0
consumed  = 0.0
rows = []
rng  = np.random.default_rng(0)
for i in range(30):
    sigma_ann = float(np.clip(rng.normal(0.7, 0.2), 0.1, None))
    rm        = 1.0 if sigma_ann < 0.8 else 0.5
    qty       = position_size(67_000, 66_700, sigma_ann,
                                risk_multiplier=rm, consumed_es=consumed,
                                es_budget=es_budget, max_position_usd=10_000.0)
    notional  = qty * 67_000
    es        = parametric_es(notional, sigma_ann)
    rejected  = consumed + es > es_budget * rc.budget_exit_threshold
    if not rejected:
        consumed += es * rng.uniform(0.4, 1.1)  # outcome ~ ES draw
    rows.append({'i': i, 'sigma_ann': sigma_ann, 'rm': rm, 'qty': qty,
                  'notional': notional, 'es': es,
                  'consumed_after': consumed, 'rejected': rejected})
seq = pd.DataFrame(rows)
seq.tail(10)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(seq['i'], seq['es'],        color='#1f77b4', label='ES per trade')
axes[0].plot(seq['i'], seq['consumed_after'], color='#d62728', label='cumulative consumed')
axes[0].axhline(es_budget,                       color='#666', ls='--', label='ES budget')
axes[0].axhline(es_budget * rc.budget_exit_threshold, color='#cc1f1a', ls=':', label='exit threshold')
axes[0].legend(); axes[0].set_ylabel('ES (USD)'); axes[0].grid(alpha=0.3)
axes[1].bar(seq['i'], seq['qty'], color=np.where(seq['rejected'], '#cc1f1a', '#1f77b4'))
axes[1].set_xlabel('trade #'); axes[1].set_ylabel('Q_final')
axes[1].grid(alpha=0.3); plt.show()

## 4. Scale-in trigger (§14.1)
Add a tranche only after EXPANSION + favourable move ≥ `scale_in_threshold_sigma · σ_P`
+ CVD slope still confirming, *and* budget headroom remains.

In [ ]:
def check_scale_in(state, scale_count, microprice, entry_price, sigma_P,
                    cvd_slope, side_sign, has_budget, cfg=rip):
    if state != 'EXPANSION':        return False
    if scale_count >= cfg.max_scale_ins: return False
    if (microprice - entry_price) * side_sign < cfg.scale_in_threshold_sigma * sigma_P:
        return False
    if cvd_slope * side_sign < cfg.scale_in_cvd_slope_min:
        return False
    return bool(has_budget)

scenarios = [
    ('expansion, big move',    dict(state='EXPANSION', scale_count=0, microprice=102.5,
                                     entry_price=100, sigma_P=1.0, cvd_slope=0.05, side_sign=+1, has_budget=True)),
    ('confirmation, too early', dict(state='CONFIRMATION', scale_count=0, microprice=102.5,
                                     entry_price=100, sigma_P=1.0, cvd_slope=0.05, side_sign=+1, has_budget=True)),
    ('expansion, cvd dying',    dict(state='EXPANSION', scale_count=0, microprice=102.5,
                                     entry_price=100, sigma_P=1.0, cvd_slope=0.005, side_sign=+1, has_budget=True)),
    ('expansion, no budget',    dict(state='EXPANSION', scale_count=0, microprice=102.5,
                                     entry_price=100, sigma_P=1.0, cvd_slope=0.05, side_sign=+1, has_budget=False)),
]
for name, kw in scenarios:
    print(f'{name:30s} → scale_in = {check_scale_in(**kw)}')

## 5. Scale-out plan (§14.2)
Pick the top-K destinations from the liquidity map (highest
`dest_score`), split the entry quantity per `scale_out_fractions`,
and tighten the stop after each fill.

In [ ]:
destinations = pd.DataFrame([
    {'price': 101.0, 'dest_score': 0.9},
    {'price': 102.5, 'dest_score': 0.7},
    {'price': 104.0, 'dest_score': 0.6},
    {'price': 105.0, 'dest_score': 0.4},  # falls below top-K
]).sort_values('dest_score', ascending=False)

def build_scale_out_plan(entry_price, side_sign, qty, destinations, cfg=rip):
    favourable = destinations[(destinations['price'] - entry_price) * side_sign > 0]
    chosen = favourable.head(len(cfg.scale_out_fractions))
    plan = []
    remaining = qty
    for i, row in enumerate(chosen.itertuples()):
        f = cfg.scale_out_fractions[i]
        q = min(qty * f, remaining)
        new_stop = entry_price if i == 0 else plan[-1]['target_price']
        plan.append({'target_price': row.price, 'qty': q, 'new_stop': new_stop})
        remaining -= q
    if remaining > 0 and plan:
        plan[-1]['qty'] += remaining
    return pd.DataFrame(plan)

build_scale_out_plan(100.0, +1, qty=3.0, destinations=destinations)

## 6. Trailing stop dynamics (§14.3)
Stop tightens monotonically once the trade enters MATURATION.

In [ ]:
def trailing_stop(prices, side_sign, sigma_P, trail_sigma=rip.trailing_stop_sigma):
    stops = []
    stop = -np.inf if side_sign > 0 else np.inf
    for p in prices:
        candidate = p - trail_sigma * sigma_P if side_sign > 0 else p + trail_sigma * sigma_P
        stop = max(stop, candidate) if side_sign > 0 else min(stop, candidate)
        stops.append(stop)
    return np.array(stops)

rng = np.random.default_rng(7)
path = 100 + np.cumsum(rng.normal(0.05, 0.4, 300))
stops = trailing_stop(path, side_sign=+1, sigma_P=1.0)
fig, ax = plt.subplots(figsize=(12, 3.4))
ax.plot(path,  color='#1f77b4', label='price')
ax.plot(stops, color='#d62728', linestyle='--', label='trailing stop')
ax.set_title('Trailing stop — only tightens, never loosens'); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## Takeaways

* The 3-step sizing formula composes cleanly: each factor scales
  monotonically and the budget clip is the final guarantee that
  Ripple cannot exceed Tide.
* `R_budget` decays smoothly from 1 to 0 as ES consumes the budget,
  so sizing tapers off *before* the hard exit threshold fires.
* Scale-in needs all four pre-conditions; scale-out is one-way.